# Evidex: fabricar fotos de autos editadas con IA

No existe un conjunto público de autos editados con IA, así que lo fabricamos:
- **Fotos:** «Car parts and car damages» de Humans in the Loop (1.812 fotos de autos, **CC0**: uso comercial libre).
- **Editores** (licencias que permiten uso comercial): SDXL inpainting (OpenRAIL++-M), Kandinsky 2.2 (Apache 2.0)
  y LaMa (Apache 2.0, el «borrador de objetos» de las galerías).
- **Ediciones:** borrar un daño real, agregar un daño a una parte sana, rehacer un foco, espejo, patente o llanta.
  Cada una con su máscara exacta (blanco = lo que cambió).

Antes de empezar, en **Settings**: **Accelerator: GPU T4 x2** e **Internet: activado**.
Opcional: **+ Add Input** → buscar «car parts and car damages» (de humansintheloop); si no, se descarga solo.

Después: **Save Version → Save & Run All (Commit)**. Tarda unas 4 a 6 horas. Al final queda
`evidex_autos_ia.zip` en **Output**: agréguelo como Input en el notebook de entrenamiento (vuelta 4).

In [ ]:
# 1. Configuración
import os
MAX_HORAS = float(os.environ.get("EVX_MAX_HORAS", 7.5))   # para antes y deja lo que alcanzó (Kaggle corta a las 12 h)
LIMITE = int(os.environ.get("EVX_LIMITE", 0))              # 0 = todas las fotos; un número = prueba rápida
SEMILLA = int(os.environ.get("EVX_SEMILLA", 0))
BASE = os.environ.get("EVX_BASE") or ("/kaggle/temp" if os.path.isdir("/kaggle") else ".")
os.makedirs(BASE, exist_ok=True)                           # en Kaggle /kaggle/temp no existe al empezar
SALIDA = f"{BASE}/autos_ia"

In [ ]:
# 2. Preparar
import subprocess, sys, glob, shutil, time
from pathlib import Path
def sh(cmd):
    print("$", cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f"Falló: {cmd}")
sh(f"{sys.executable} -m pip install -q diffusers transformers accelerate")
Path("training").mkdir(exist_ok=True)
ARCHIVOS = {
 "generar_autos.py": "\"\"\"Genera fotos de autos editadas con IA, con la máscara exacta de lo cambiado, para entrenar a Evidex.\n\nNo existe un conjunto público de autos editados con IA. Este script lo fabrica:\n  - Fotos: «Car parts and car damages» de Humans in the Loop (1.812 fotos, CC0 1.0: dominio público, uso comercial\n    libre). Trae polígonos de 21 partes del auto y 8 tipos de daño.\n  - Editores (licencias que permiten uso comercial):\n      sdxl       SDXL inpainting 0.1 (diffusers, OpenRAIL++-M: respetar sus restricciones de uso)\n      kandinsky  Kandinsky 2.2 inpainting (Apache 2.0)\n      lama       LaMa (Apache 2.0), un «borrador de objetos» como el de las galerías de los teléfonos\n  - Ediciones, como las haría un asegurado:\n      borrar_dano    se borra un daño real (abolladura, rayón, trizadura...)\n      agregar_dano   se agrega un daño a una parte sana (puerta, tapabarro, parachoques, capó...)\n      cambiar_parte  se rehace un foco, espejo, patente, parrilla o llanta\n    Igual que en el teléfono, la IA trabaja sobre un recorte alrededor de la zona y el resultado se pega en la foto\n    original: fuera de la máscara los píxeles quedan idénticos.\n\nSalida (formato de fotos propias de training/datos.py):\n  salida/originales/hitl0001.jpg          la foto tal cual\n  salida/editadas/hitl0001_agregar_dano_sdxl.png\n  salida/mascaras/hitl0001_agregar_dano_sdxl.png   (blanco = cambiado)\n  salida/registro.jsonl                   qué se hizo en cada una (editor, prompt, parte, semilla)\n  salida/licencia.txt\n\n  python training/generar_autos.py --hitl datos/hitl --salida datos/autos_ia --editores sdxl,lama --parte 0 --partes 2\nNecesita GPU para sdxl y kandinsky; lama corre también en CPU.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\n\nDANOS = {\"Missing part\", \"Broken part\", \"Scratch\", \"Cracked\", \"Dent\", \"Flaking\", \"Paint chip\", \"Corrosion\"}\nCARROCERIA = {\"Front-door\", \"Back-door\", \"Fender\", \"Quarter-panel\", \"Hood\", \"Trunk\", \"Front-bumper\", \"Back-bumper\",\n              \"Rocker-panel\", \"Roof\"}\nPIEZAS = {\"Headlight\": \"a broken shattered car headlight\", \"Tail-light\": \"a cracked broken car tail light\",\n          \"Mirror\": \"a broken car side mirror hanging\", \"License-plate\": \"a car license plate\",\n          \"Grille\": \"a damaged cracked car front grille\", \"Front-wheel\": \"a car wheel with a scratched bent rim\",\n          \"Back-wheel\": \"a car wheel with a scratched bent rim\"}\nDANO_PROMPT = [\"a deep dent in the car {p}, crumpled metal\", \"long deep scratches on the car {p} paint\",\n               \"cracked paint and a dent on the car {p}\", \"a crushed damaged car {p} after a collision\",\n               \"a broken cracked plastic car {p}\"]\nLIMPIO = \"clean undamaged car {p}, smooth glossy paint, no dents, no scratches\"\nNEGATIVO = \"text, watermark, logo, cartoon, drawing, blurry, low quality\"\nPARTES_ES = {\"Front-door\": \"door\", \"Back-door\": \"door\", \"Fender\": \"fender\", \"Quarter-panel\": \"quarter panel\",\n             \"Hood\": \"hood\", \"Trunk\": \"trunk\", \"Front-bumper\": \"front bumper\", \"Back-bumper\": \"rear bumper\",\n             \"Rocker-panel\": \"side skirt\", \"Roof\": \"roof\"}\nLICENCIA = (\"Fotos: Humans in the Loop «Car parts and car damages», CC0 1.0 (dominio público). \"\n            \"Ediciones: SDXL inpainting 0.1 (OpenRAIL++-M), Kandinsky 2.2 (Apache 2.0), LaMa (Apache 2.0)\")\nGENERATIVOS = (\"sdxl\", \"kandinsky\")\n\n\n# ---- fotos y anotaciones (formato Supervisely: img/<foto> + ann/<foto>.json) -------------------------------\ndef listar(raiz: Path) -> list[dict]:\n    items = []\n    for ann in sorted(raiz.rglob(\"ann/*.json\")):\n        img = ann.parent.parent / \"img\" / ann.name[:-5]\n        if not img.exists():\n            continue\n        try:\n            data = json.loads(ann.read_text(encoding=\"utf-8\"))\n        except (OSError, ValueError):\n            continue\n        objs = [{\"clase\": o.get(\"classTitle\"), \"poligono\": [tuple(p) for p in (o.get(\"points\") or {}).get(\"exterior\") or []]}\n                for o in data.get(\"objects\") or [] if o.get(\"geometryType\") == \"polygon\"]\n        objs = [o for o in objs if len(o[\"poligono\"]) >= 3]\n        if objs:\n            items.append({\"img\": img, \"objetos\": objs})\n    return items\n\n\ndef mascara_poligono(tam, poligono, dilatar: int = 0) -> Image.Image:\n    m = Image.new(\"L\", tam, 0)\n    ImageDraw.Draw(m).polygon(poligono, fill=255)\n    if dilatar:\n        m = m.filter(ImageFilter.MaxFilter(dilatar * 2 + 1))\n    return m\n\n\ndef mancha_dentro(tam, poligono, rng: random.Random) -> Image.Image:\n    \"\"\"Una zona irregular dentro de una parte de la carrocería (donde se «agrega» un daño).\"\"\"\n    parte = np.asarray(mascara_poligono(tam, poligono)) > 0\n    ys, xs = np.nonzero(parte)\n    if len(xs) < 50:\n        return Image.new(\"L\", tam, 0)\n    w, h = xs.max() - xs.min(), ys.max() - ys.min()\n    k = rng.randrange(len(xs))\n    cx, cy = xs[k], ys[k]\n    m = Image.new(\"L\", tam, 0)\n    d = ImageDraw.Draw(m)\n    for _ in range(rng.randint(2, 4)):                       # varias elipses = forma irregular\n        rx, ry = w * rng.uniform(.08, .22), h * rng.uniform(.08, .22)\n        ox, oy = rng.uniform(-rx, rx) * .6, rng.uniform(-ry, ry) * .6\n        d.ellipse([cx + ox - rx, cy + oy - ry, cx + ox + rx, cy + oy + ry], fill=255)\n    return Image.fromarray(((np.asarray(m) > 0) & parte).astype(np.uint8) * 255)\n\n\ndef planificar(item: dict, idx: int, editores: list[str], semilla: int) -> list[dict]:\n    \"\"\"Qué ediciones se le hacen a una foto (siempre las mismas para la misma foto y semilla).\"\"\"\n    rng = random.Random(semilla * 100003 + idx)\n    gen = [e for e in editores if e in GENERATIVOS]\n    danos = [o for o in item[\"objetos\"] if o[\"clase\"] in DANOS]\n    cuerpo = [o for o in item[\"objetos\"] if o[\"clase\"] in CARROCERIA]\n    piezas = [o for o in item[\"objetos\"] if o[\"clase\"] in PIEZAS]\n    trabajos = []\n    if danos:\n        o = rng.choice(danos)\n        borrador = \"lama\" if \"lama\" in editores and (not gen or rng.random() < .5) else (rng.choice(gen) if gen else None)\n        if borrador:\n            parte = rng.choice([PARTES_ES[c[\"clase\"]] for c in cuerpo]) if cuerpo else \"body panel\"\n            trabajos.append({\"tipo\": \"borrar_dano\", \"editor\": borrador, \"clase\": o[\"clase\"], \"poligono\": o[\"poligono\"],\n                             \"prompt\": LIMPIO.format(p=parte)})\n    if cuerpo and gen:\n        o = rng.choice(cuerpo)\n        trabajos.append({\"tipo\": \"agregar_dano\", \"editor\": rng.choice(gen), \"clase\": o[\"clase\"], \"poligono\": o[\"poligono\"],\n                         \"prompt\": rng.choice(DANO_PROMPT).format(p=PARTES_ES[o[\"clase\"]])})\n    if piezas and gen and rng.random() < .35:\n        o = rng.choice(piezas)\n        trabajos.append({\"tipo\": \"cambiar_parte\", \"editor\": rng.choice(gen), \"clase\": o[\"clase\"], \"poligono\": o[\"poligono\"],\n                         \"prompt\": PIEZAS[o[\"clase\"]]})\n    for t in trabajos:\n        t[\"semilla\"] = rng.randrange(2 ** 31)\n    return trabajos\n\n\ndef mascara_de(trabajo: dict, tam) -> Image.Image:\n    if trabajo[\"tipo\"] == \"agregar_dano\":\n        return mancha_dentro(tam, trabajo[\"poligono\"], random.Random(trabajo[\"semilla\"]))\n    return mascara_poligono(tam, trabajo[\"poligono\"], dilatar=max(3, min(tam) // 120))\n\n\ndef ventana(mask: np.ndarray, lado: int):\n    \"\"\"Recorte cuadrado alrededor de la zona (lo que hace el teléfono antes de mandar a la IA).\"\"\"\n    h, w = mask.shape\n    ys, xs = np.nonzero(mask)\n    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1\n    s = int(max(x1 - x0, y1 - y0) * 1.6)\n    s = min(max(s, min(lado, w, h)), w, h)\n    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2\n    bx = int(min(max(cx - s // 2, 0), w - s))\n    by = int(min(max(cy - s // 2, 0), h - s))\n    return bx, by, bx + s, by + s\n\n\ndef pegar(orig: Image.Image, gen: Image.Image, mask: Image.Image, caja) -> tuple[Image.Image, Image.Image]:\n    \"\"\"Pega el recorte editado sobre la foto original, solo dentro de la máscara (borde suavizado).\"\"\"\n    alfa = mask.crop(caja).filter(ImageFilter.GaussianBlur(2))\n    out = orig.copy()\n    out.paste(gen.resize(alfa.size, Image.LANCZOS), caja[:2], alfa)\n    final = Image.new(\"L\", orig.size, 0)\n    final.paste(alfa.point(lambda v: 255 if v > 0 else 0), caja[:2])        # todo píxel que cambió, aunque sea poco\n    return out, final\n\n\n# ---- editores ------------------------------------------------------------------------------------------\nclass Lama:\n    URL = \"https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt\"\n    lado = 512\n\n    def __init__(self, dispositivo: str, modelo: Path | None = None):\n        import torch\n        import urllib.request\n        self.torch, self.disp = torch, dispositivo\n        if modelo is None or not Path(modelo).exists():\n            modelo = Path(modelo or \"big-lama.pt\")\n            if not modelo.exists():\n                urllib.request.urlretrieve(self.URL, modelo)\n        self.m = torch.jit.load(str(modelo), map_location=dispositivo).eval()\n\n    def __call__(self, img: Image.Image, mask: Image.Image, prompt: str, semilla: int) -> Image.Image:\n        t = self.torch\n        w, h = img.size\n        W, H = (w + 7) // 8 * 8, (h + 7) // 8 * 8\n        a = np.zeros((H, W, 3), np.float32)\n        a[:h, :w] = np.asarray(img, np.float32) / 255\n        m = np.zeros((H, W), np.float32)\n        m[:h, :w] = (np.asarray(mask) > 0)\n        with t.no_grad():\n            out = self.m(t.from_numpy(a.transpose(2, 0, 1))[None].to(self.disp),\n                         t.from_numpy(m)[None, None].to(self.disp))\n        r = np.clip(out[0].permute(1, 2, 0).cpu().numpy() * 255, 0, 255).astype(np.uint8)\n        return Image.fromarray(r[:h, :w])\n\n\nclass Difusor:\n    def __init__(self, nombre: str, dispositivo: str):\n        import torch\n        from diffusers import AutoPipelineForInpainting\n        self.torch, self.disp = torch, dispositivo\n        repo, self.lado, kw = {\n            \"sdxl\": (\"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\", 1024, {\"variant\": \"fp16\"}),\n            \"kandinsky\": (\"kandinsky-community/kandinsky-2-2-decoder-inpaint\", 768, {}),\n        }[nombre]\n        self.nombre = nombre\n        import os\n        self.lado = int(os.environ.get(\"EVX_LADO_GEN\", self.lado))           # solo para pruebas en CPU\n        self.pasos = int(os.environ.get(\"EVX_PASOS_GEN\", 25))\n        dtype = torch.float16 if dispositivo == \"cuda\" else torch.float32\n        if dtype == torch.float32:\n            kw = {}\n        self.pipe = AutoPipelineForInpainting.from_pretrained(repo, torch_dtype=dtype, **kw).to(dispositivo)\n        self.pipe.set_progress_bar_config(disable=True)\n\n    def __call__(self, img: Image.Image, mask: Image.Image, prompt: str, semilla: int) -> Image.Image:\n        L = self.lado\n        g = self.torch.Generator(device=self.disp).manual_seed(semilla)\n        kw = {\"strength\": .99} if self.nombre == \"sdxl\" else {}\n        out = self.pipe(prompt=prompt, negative_prompt=NEGATIVO, image=img.resize((L, L), Image.LANCZOS),\n                        mask_image=mask.resize((L, L), Image.NEAREST), height=L, width=L,\n                        num_inference_steps=self.pasos, guidance_scale=7.5, generator=g, **kw).images[0]\n        return out.resize(img.size, Image.LANCZOS)\n\n\ndef cargar_editor(nombre: str, dispositivo: str, lama_modelo: Path | None = None):\n    return Lama(dispositivo, lama_modelo) if nombre == \"lama\" else Difusor(nombre, dispositivo)\n\n\n# ---- todo junto ----------------------------------------------------------------------------------------\ndef generar(items: list[dict], salida: Path, editores: dict, parte: int = 0, partes: int = 1, semilla: int = 0,\n            max_horas: float = 0, limite: int = 0) -> int:\n    for d in (\"originales\", \"editadas\", \"mascaras\"):\n        (salida / d).mkdir(parents=True, exist_ok=True)\n    (salida / \"licencia.txt\").write_text(LICENCIA, encoding=\"utf-8\")\n    inicio, hechas = time.time(), 0\n    with (salida / \"registro.jsonl\").open(\"a\", encoding=\"utf-8\") as reg:\n        for idx, item in enumerate(items):\n            if idx % partes != parte:\n                continue\n            if (max_horas and time.time() - inicio > max_horas * 3600) or (limite and hechas >= limite):\n                break\n            base = f\"hitl{idx:04d}\"\n            trabajos = planificar(item, idx, list(editores), semilla)\n            if not trabajos:\n                continue\n            try:\n                orig = Image.open(item[\"img\"]).convert(\"RGB\")\n            except OSError:\n                continue\n            dst = salida / \"originales\" / (base + item[\"img\"].suffix.lower())\n            if not dst.exists():\n                shutil.copyfile(item[\"img\"], dst)\n            for t in trabajos:\n                stem = f\"{base}_{t['tipo']}_{t['editor']}\"\n                if (salida / \"editadas\" / f\"{stem}.png\").exists():\n                    continue                                      # ya hecha (se puede retomar)\n                mask = mascara_de(t, orig.size)\n                frac = (np.asarray(mask) > 0).mean()\n                if not .002 <= frac <= .35:\n                    continue\n                editor = editores[t[\"editor\"]]\n                caja = ventana(np.asarray(mask) > 0, getattr(editor, \"lado\", 512))\n                gen = editor(orig.crop(caja), mask.crop(caja), t[\"prompt\"], t[\"semilla\"])\n                editada, final = pegar(orig, gen, mask, caja)\n                editada.save(salida / \"editadas\" / f\"{stem}.png\")\n                final.save(salida / \"mascaras\" / f\"{stem}.png\")\n                reg.write(json.dumps({\"editada\": f\"{stem}.png\", \"original\": dst.name, \"foto_fuente\": item[\"img\"].name,\n                                      **{k: t[k] for k in (\"tipo\", \"editor\", \"clase\", \"prompt\", \"semilla\")},\n                                      \"area\": round(float(frac), 4)}, ensure_ascii=False) + \"\\n\")\n                reg.flush()\n                hechas += 1\n    return hechas\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--hitl\", type=Path, required=True, help=\"carpeta del conjunto de Humans in the Loop (descomprimido)\")\n    ap.add_argument(\"--salida\", type=Path, required=True)\n    ap.add_argument(\"--editores\", default=\"sdxl,kandinsky,lama\")\n    ap.add_argument(\"--parte\", type=int, default=0, help=\"para repartir entre varias GPU: esta parte...\")\n    ap.add_argument(\"--partes\", type=int, default=1, help=\"...de cuántas\")\n    ap.add_argument(\"--semilla\", type=int, default=0)\n    ap.add_argument(\"--max-horas\", type=float, default=0)\n    ap.add_argument(\"--limite\", type=int, default=0, help=\"como máximo N ediciones (prueba rápida)\")\n    ap.add_argument(\"--lama-modelo\", type=Path)\n    a = ap.parse_args(argv)\n    import torch\n    disp = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    items = listar(a.hitl)\n    print(f\"{len(items)} fotos con anotaciones; editores {a.editores}; parte {a.parte}/{a.partes}; {disp}\", flush=True)\n    editores = {e: cargar_editor(e, disp, a.lama_modelo) for e in a.editores.split(\",\") if e}\n    t = time.time()\n    n = generar(items, a.salida, editores, a.parte, a.partes, a.semilla, a.max_horas, a.limite)\n    print(f\"Listo: {n} ediciones en {time.time() - t:.0f} s -> {a.salida}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for nombre, contenido in ARCHIVOS.items():
    Path("training", nombre).write_text(contenido, encoding="utf-8")
import torch
N_GPU = torch.cuda.device_count()
LAMA = f"{BASE}/big-lama.pt"            # LaMa se baja una vez aquí (las dos GPU lo comparten)
if not os.path.exists(LAMA):
    sh(f"curl -fsSL --retry 3 --create-dirs -o {LAMA} https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt")
print("GPU:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NO HAY GPU (active el acelerador en Settings)")

In [ ]:
# 3. Las fotos de autos (CC0): del Input si se agregó, si no se descargan
hitl = sorted({str(Path(p).parents[2]) for p in glob.glob("/kaggle/input/**/ann/*.json", recursive=True)})
if hitl:
    HITL = os.path.commonpath(hitl)
else:
    HITL = f"{BASE}/hitl"
    if not glob.glob(f"{HITL}/**/ann/*.json", recursive=True):
        sh(f"curl -fsSL --retry 3 --create-dirs -o {BASE}/hitl.zip https://www.kaggle.com/api/v1/datasets/download/humansintheloop/car-parts-and-car-damages")
        sh(f"cd {BASE} && mkdir -p hitl && cd hitl && unzip -q ../hitl.zip -x '*/masks_human/*' '*/masks_machine/*'")
print("Fotos de autos en:", HITL, "-", len(glob.glob(f"{HITL}/**/img/*", recursive=True)), "fotos")

In [ ]:
# 4. Generar (en paralelo: una GPU con SDXL y otra con Kandinsky; LaMa en las dos)
os.makedirs(SALIDA, exist_ok=True)
planes = [("sdxl,lama", 0), ("kandinsky,lama", 1)] if N_GPU >= 2 else [(os.environ.get("EVX_EDITORES", "sdxl,lama"), 0)]
procs = []
for i, (editores, gpu) in enumerate(planes):
    log = open(f"{BASE}/generar_{i}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    cmd = [sys.executable, "training/generar_autos.py", "--hitl", HITL, "--salida", SALIDA, "--editores", editores,
           "--parte", str(i), "--partes", str(len(planes)), "--semilla", str(SEMILLA), "--max-horas", str(MAX_HORAS),
           "--lama-modelo", LAMA] + (["--limite", str(LIMITE)] if LIMITE else [])
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env))
while any(p.poll() is None for p in procs):
    time.sleep(300)
    n = len(os.listdir(f"{SALIDA}/editadas")) if os.path.isdir(f"{SALIDA}/editadas") else 0
    print(time.strftime("%H:%M"), f"{n} fotos editadas", flush=True)
for i, p in enumerate(procs):
    print(f"--- proceso {i} (código {p.returncode}) ---")
    print("".join(open(f"{BASE}/generar_{i}.log").readlines()[-5:]))
if any(p.returncode for p in procs):
    raise SystemExit("Un proceso falló: mire el registro arriba.")

In [ ]:
# 5. Revisar a ojo: 8 ejemplos (original | editada | máscara)
import json, random
from PIL import Image
reg = [json.loads(x) for x in open(f"{SALIDA}/registro.jsonl", encoding="utf-8")]
print(len(reg), "ediciones:", {t: sum(r["tipo"] == t for r in reg) for t in ("borrar_dano", "agregar_dano", "cambiar_parte")},
      {e: sum(r["editor"] == e for r in reg) for e in ("sdxl", "kandinsky", "lama")})
filas = []
for r in random.Random(1).sample(reg, min(8, len(reg))):
    ims = [Image.open(f"{SALIDA}/originales/{r['original']}").convert("RGB"),
           Image.open(f"{SALIDA}/editadas/{r['editada']}").convert("RGB"),
           Image.open(f"{SALIDA}/mascaras/{r['editada']}").convert("RGB")]
    ims = [im.resize((int(im.width * 240 / im.height), 240)) for im in ims]
    fila = Image.new("RGB", (sum(i.width for i in ims) + 20, 240), "white")
    x = 0
    for im in ims:
        fila.paste(im, (x, 0)); x += im.width + 10
    filas.append(fila)
hoja = Image.new("RGB", (max(f.width for f in filas), 250 * len(filas)), "white")
for i, f in enumerate(filas):
    hoja.paste(f, (0, 250 * i))
os.makedirs("/kaggle/working" if os.path.isdir("/kaggle") else ".", exist_ok=True)
hoja.save(("/kaggle/working/" if os.path.isdir("/kaggle") else "") + "muestra_autos_ia.jpg", quality=85)
hoja

In [ ]:
# 6. Empaquetar (Output → evidex_autos_ia.zip; agréguelo como Input del notebook de entrenamiento)
destino = "/kaggle/working/evidex_autos_ia" if os.path.isdir("/kaggle") else "evidex_autos_ia"
shutil.make_archive(destino, "zip", SALIDA)
print("Listo:", destino + ".zip", round(os.path.getsize(destino + ".zip") / 1e9, 2), "GB")